# COMP 3404 - Final Project -  Inforgraphic 

### By: Hageer Birama - 202034484 & Nicole Bishop-Adigwe - 201960085
### Due: Sunday April 12, 2026

#### Import dependencies

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.stats import gaussian_kde
import json, urllib.request, os

%config InlineBackend.figure_format = 'retina'

#### Load data and preprocess it

In [2]:
# load dataset
df = pd.read_csv('birds.csv')

# remove extra spaces from column names
df.columns = df.columns.str.strip()

In [3]:
# format observation date into datetime 
df['OBSERVATION DATE'] = pd.to_datetime(df['OBSERVATION DATE'], errors='coerce')
df = df.dropna(subset=['OBSERVATION DATE'])     # remove rows where date it missing

# extract month number and month name for grouping
df['month_num']  = df['OBSERVATION DATE'].dt.month
df['month_name'] = df['OBSERVATION DATE'].dt.strftime('%b')

# convert the Xs to 1s
df['OBSERVATION COUNT'] = pd.to_numeric(
    df['OBSERVATION COUNT'].replace('X', 1), errors='coerce'
).fillna(1)

# remove rows without LAT or LONG
df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])

print("Data preprocessed.") 

Data preprocessed.


In [4]:
# order of months
MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

##### Data for Heatmap 

In [5]:
# only display top 15 bird species in heatmap
top15_species = (
    df.groupby('COMMON NAME')['OBSERVATION COUNT']
    .sum().nlargest(15).index.tolist()
)

# create pivot table (species vs month)
heatmap_df = (
    df[df['COMMON NAME'].isin(top15_species)]
    .groupby(['COMMON NAME','month_name'])['OBSERVATION COUNT'].sum()
    .reset_index()
    .pivot(index='COMMON NAME', columns='month_name', values='OBSERVATION COUNT')
    .reindex(columns=MONTH_ORDER).fillna(0)
)

# sort based on peak activity month
heatmap_df['_peak'] = heatmap_df.idxmax(axis=1).map({m:i for i,m in enumerate(MONTH_ORDER)})
heatmap_df = heatmap_df.sort_values('_peak').drop(columns='_peak')

print('Heatmap:', heatmap_df.shape)

Heatmap: (15, 12)


##### Data for Geographical Scatter

In [6]:
# filter to NL on map
geo_df = df[
    df['LATITUDE'].between(46.5, 60.5) &
    df['LONGITUDE'].between(-67.5, -52.0)
].copy()

# aggregate observations based on locality
loc_df = (
    geo_df.groupby(['LOCALITY','LATITUDE','LONGITUDE'])
    .agg(total_obs=('OBSERVATION COUNT','sum'),
         num_checklist=('GLOBAL UNIQUE IDENTIFIER','count'))
    .reset_index()
)

print(f'Localities: {len(loc_df):,}')

Localities: 6,577


##### Data for Violin 

In [7]:
TOP_N = 15       # use top 15 bird types

species_totals = (
    df.groupby('COMMON NAME')['OBSERVATION COUNT'].sum()
    .nlargest(TOP_N).reset_index()
    .rename(columns={'OBSERVATION COUNT':'total'})
)

# sampling data for violin w/ limit of 400/species 
violin_df = (
    df[df['COMMON NAME'].isin(species_totals['COMMON NAME'])]
    .groupby('COMMON NAME')
    .apply(lambda g: g.sample(min(len(g), 400), random_state=42))
    .reset_index(drop=True)
)

print(f'Violin rows: {len(violin_df):,}')

Violin rows: 5,859


#### Load NL map via GeoJSON file

In [8]:
NL_GEOJSON_URL = (
    'https://raw.githubusercontent.com/codeforgermany/click_that_hood/'
    'main/public/data/canada.geojson'
)

GEOJSON_CACHE = 'canada_provinces.geojson'

nl_polygons   = []   # list of (lons, lats) for NL
other_polys   = []   # neighbouring provinces 

try:
    # download file if not already downloaded
    if not os.path.exists(GEOJSON_CACHE):
        print('Downloading GeoJSON …')
        urllib.request.urlretrieve(NL_GEOJSON_URL, GEOJSON_CACHE)
        
    with open(GEOJSON_CACHE) as fh:
        gj = json.load(fh)

    # extracts ploygon coordinates
    def extract_polys(feat):
        polys = []
        geom  = feat['geometry']
        
        if geom['type'] == 'Polygon':
            coords = geom['coordinates'][0]
            polys.append(([c[0] for c in coords], [c[1] for c in coords]))
            
        elif geom['type'] == 'MultiPolygon':
            for poly in geom['coordinates']:
                coords = poly[0]
                polys.append(([c[0] for c in coords], [c[1] for c in coords]))
                
        return polys

    # separate NL from surrounding provinces
    for feat in gj['features']:
        name = feat['properties'].get('name', '')
        
        if 'Newfoundland' in name or 'Labrador' in name:
            nl_polygons.extend(extract_polys(feat))
            
        elif name in ('Quebec', 'Nova Scotia', 'New Brunswick','Prince Edward Island'):
            other_polys.extend(extract_polys(feat))

    print(f'NL polygons: {len(nl_polygons)}  |  Neighbouring: {len(other_polys)}')
    
except Exception as e:
    print(f'GeoJSON load failed ({e}). Coast lines will be omitted.')

NL polygons: 21  |  Neighbouring: 18


In [9]:
# draws the map of NL
def draw_nl_map(ax):
    # draw surrounding privnces as lighter than NL
    for lons, lats in other_polys:
        ax.fill(lons, lats, color='#c8d8c0', alpha=0.55, zorder=1)
        ax.plot(lons, lats, color='#99aaa0', lw=0.4, alpha=0.7, zorder=2)
        
    # draw NL
    for lons, lats in nl_polygons:
        ax.fill(lons, lats, color='#b8d4a0', alpha=0.85, zorder=3)
        ax.plot(lons, lats, color='#6a9a72', lw=0.6, alpha=0.90, zorder=4)

#### Colour theme for infographic 

In [10]:
# colours for background, accents, titles, texts, map 
BG_DARK   = '#0D1B2A'
BG_PANEL  = '#112233'
ACCENT1   = '#F4A261'   # amber
ACCENT2   = '#2A9D8F'   # teal
ACCENT3   = '#E9C46A'   # gold
ACCENT4   = '#E76F51'   # orange-y
TEXT_MAIN = '#F0EAD6'
TEXT_DIM  = '#8899AA'
GRID_CLR  = '#1E3048'
OCEAN_CLR = '#a8d4e6'   # light blue

In [11]:
# gradient for heatmap 
hm_cmap  = LinearSegmentedColormap.from_list(
    'nl_birds', ['#0D1B2A','#1a4060','#2A9D8F','#F4A261','#E76F51'], N=256)

# gradient for scatter
geo_cmap = LinearSegmentedColormap.from_list(
    'geo', ['#1a0050','#8B008B','#FF8C00','#FFD700'], N=256)  # purple→orange→gold like ref image

#### Infographic Figure Formatting

In [12]:
# define infographic according to sepsification (24x36 in)
DPI = 100
FIG_W, FIG_H = 24, 36

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI, facecolor=BG_DARK)

<Figure size 2400x3600 with 0 Axes>

In [13]:
# Fig layout = header + heatmap + scatter + violin
gs = gridspec.GridSpec(
    4, 2,
    figure=fig,
    height_ratios=[0.07, 1.0, 1.10, 0.95],
    hspace=0.13,
    wspace=0.06,
    left=0.045, right=0.955,
    top=0.975, bottom=0.018
)

##### Header 

In [14]:
ax_hdr = fig.add_subplot(gs[0, :])
ax_hdr.set_facecolor(BG_DARK)
ax_hdr.axis('off')

# infographic titles
ax_hdr.text(
    0.5, 0.85,
    ' ~ BIRDS OF NEWFOUNDLAND & LABRADOR ~ ',
    ha='center', va='top', transform=ax_hdr.transAxes,
    fontsize=38, fontweight='bold', color=ACCENT1, fontfamily='serif')

ax_hdr.text(
    0.5, 0.10,
    'A visual exploration of 2025 bird observations in Newfoundland & Labrador',
    ha='center', va='top', transform=ax_hdr.transAxes,
    fontsize=17, color=TEXT_DIM, fontstyle='italic')

#ax_hdr.axhline(0.001, color=ACCENT2, linewidth=2, xmin=0.05, xmax=0.95)

print('Header done.')

Header done.


#### Visualizations

##### Heatmap

In [15]:
# create axis 
ax1 = fig.add_subplot(gs[1, :])
ax1.set_facecolor(BG_PANEL)       # background colour

# use log scale -> reduces variability b/w values
log_data = np.log1p(heatmap_df.values)

# draw heatmap
im = ax1.imshow(log_data, aspect='auto', cmap=hm_cmap, interpolation='nearest')

# add size to each cell
for r, species in enumerate(heatmap_df.index):
    for c, month in enumerate(MONTH_ORDER):
        val = heatmap_df.loc[species, month]

        # only label cells that have a value
        if val > 0:
            txt_color = TEXT_MAIN if log_data[r, c] < log_data.max() * 0.6 else BG_DARK    # text colour for cells
            
            ax1.text(
                c, r,
                f'{int(val):,}' if val < 10000 else f'{val/1000:.0f}k',
                ha='center', va='center',
                fontsize=7.5, color=txt_color, fontweight='semibold')

# title and styling 
ax1.set_xticks(range(12))
ax1.set_xticklabels(MONTH_ORDER, fontsize=14, color=TEXT_MAIN)

ax1.set_yticks(range(len(heatmap_df)))
ax1.set_yticklabels(heatmap_df.index, fontsize=12.5, color=TEXT_MAIN)

ax1.tick_params(colors=TEXT_DIM, length=0)
for spine in ax1.spines.values():
    spine.set_edgecolor(GRID_CLR)

# colour bar on side of figure
cbar = fig.colorbar(im, ax=ax1, orientation='vertical', pad=0.01, fraction=0.015)
cbar.set_label('Observation count (log scale)', color=TEXT_DIM, fontsize=11)
cbar.ax.yaxis.set_tick_params(color=TEXT_DIM, labelcolor=TEXT_DIM, labelsize=9)
cbar.outline.set_edgecolor(GRID_CLR)

ax1.set_title(
    '1. When do Birds Appear?   —   Monthly Observations for Top 15 Species',
    fontsize=20, fontweight='bold', color=ACCENT3, pad=14, loc='left')

ax1.text(
    0.0, -0.045,
    'Warmer colours indicate more sightings. Species sorted by their peak activity month.',
    transform=ax1.transAxes, fontsize=11, color=TEXT_DIM, ha='left')

print('Viz 1 done.')

Viz 1 done.


##### Geographical Scatter

In [16]:
# create axis
ax2 = fig.add_subplot(gs[2, :])

ax2.set_facecolor(OCEAN_CLR)   # light blue ocean

# draw NL
draw_nl_map(ax2)

# scale points based on number of observations
sizes  = np.clip(np.sqrt(loc_df['num_checklist']) * 7, 8, 600)

# colour points based off total observations & scale them
colors = np.log1p(loc_df['total_obs'])

# plot the scattered points
sc = ax2.scatter(
    loc_df['LONGITUDE'], loc_df['LATITUDE'],
    c=colors, s=sizes, cmap=geo_cmap,
    alpha=0.82, linewidths=0.35,
    edgecolors='#ffffff60', zorder=5)

# using denisty contours to show hotspots via KDE
try:
    # sample subset 
    kde_sample = geo_df.sample(min(len(geo_df), 6000), random_state=42)
    
    xy  = np.vstack([kde_sample['LONGITUDE'], kde_sample['LATITUDE']])
    kde = gaussian_kde(xy, bw_method=0.10)

    # make grid for contour calculation
    lon_g = np.linspace(geo_df['LONGITUDE'].min()-0.3, geo_df['LONGITUDE'].max()+0.3, 220)
    lat_g = np.linspace(geo_df['LATITUDE'].min()-0.3,  geo_df['LATITUDE'].max()+0.3,  220)
    
    LON, LAT = np.meshgrid(lon_g, lat_g)

    # evaluate desnity 
    Z = kde(np.vstack([LON.ravel(), LAT.ravel()])).reshape(LON.shape)

    # draw lines
    ax2.contour(LON, LAT, Z, levels=5, colors='#555555',
                alpha=0.20, linewidths=0.6, zorder=6)
    
    print('KDE done.')
    
except Exception as e:
    print(f'KDE skipped: {e}')

# label top 5 in filtered selection
top_locs = loc_df.nlargest(5, 'num_checklist').reset_index(drop=True)

# ensure they dont overlap
OFFSETS  = [(0.7, 0.6), (-2.8, 0.9), (0.6, -0.9), (-2.2, -1.0), (1.1, -0.6)]

for i, row in top_locs.iterrows():
    label = (row['LOCALITY'].split('--')[-1]
             .replace(' - ', '\n').strip()[:35])
    
    dx, dy = OFFSETS[i % len(OFFSETS)]
    
    ax2.annotate(
        label,
        xy=(row['LONGITUDE'], row['LATITUDE']),
        xytext=(row['LONGITUDE']+dx, row['LATITUDE']+dy),
        fontsize=9.5, color='#1a1a1a', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#333333', lw=1.1),
        bbox=dict(boxstyle='round,pad=0.25', fc='#ffffffcc',
                  ec='#6a9a72', lw=0.9),
        zorder=8)

# title and styling 
ax2.set_xlim(geo_df['LONGITUDE'].min()-1.0, geo_df['LONGITUDE'].max()+1.0)
ax2.set_ylim(geo_df['LATITUDE'].min()-0.8,  geo_df['LATITUDE'].max()+0.8)

ax2.set_xlabel('Longitude', fontsize=12, color='#444444')
ax2.set_ylabel('Latitude',  fontsize=12, color='#444444')

ax2.tick_params(colors='#555555', labelsize=10)

for spine in ax2.spines.values():
    spine.set_edgecolor('#999999')
    
ax2.grid(color='#cccccc', linewidth=0.35, alpha=0.6, linestyle='--')

# colour bar for desnity/intensity 
cbar2 = fig.colorbar(sc, ax=ax2, orientation='vertical', pad=0.01, fraction=0.015)

cbar2.set_label('Total obs. (log)', color=TEXT_DIM, fontsize=11)
cbar2.ax.yaxis.set_tick_params(color=TEXT_DIM, labelcolor=TEXT_DIM, labelsize=9)
cbar2.outline.set_edgecolor('#999999')

# legend for size of points
for n_check, label in [(5, '5 checklists'), (50, '50'), (200, '200+')]:
    ax2.scatter([], [], s=np.clip(np.sqrt(n_check)*7, 8, 600),
                c='#888888', alpha=0.75, label=label)
    
legend = ax2.legend(
    title='Checklists submitted', title_fontsize=10,
    fontsize=10, loc='lower left',
    framealpha=0.80, facecolor='#ffffffdd',
    edgecolor='#999999', labelcolor='#222222')

legend.get_title().set_color('#444444')

ax2.set_title(
    '2. Where to go for Bird Watching?   —   Birdwatching Hotspots Across Newfoundland & Labrador',
    fontsize=20, fontweight='bold', color=ACCENT3, pad=14, loc='left')

ax2.text(
    0.0, -0.045,
    'Each point is a unique locality. Size = number of checklists submitted, colour = total birds observed. '
    'Contours highlight areas of highest observer density.',
    transform=ax2.transAxes, fontsize=11, color=TEXT_DIM, ha='left')

print('Viz 2 done.')

KDE done.
Viz 2 done.


##### Violin 

In [17]:
# create axis 
ax3 = fig.add_subplot(gs[3, :])
ax3.set_facecolor(BG_PANEL)

# sort species by median observation count (use total or mean??)
median_counts = (
    violin_df.groupby('COMMON NAME')['OBSERVATION COUNT']
    .median().reindex(species_totals['COMMON NAME'])
)

species_order = median_counts.sort_values(ascending=False).index.tolist()

# reverse order so highest = top
species_order = species_order[::-1]

# map species to location on y
sp_to_y = {sp: i for i, sp in enumerate(species_order)}

# number of plots to display
n_sp = len(species_order)

# make list of data for each species
violin_data = [
    violin_df[violin_df['COMMON NAME'] == sp]['OBSERVATION COUNT'].values
    for sp in species_order
]

import seaborn as sns


# violin plot colours
viol_colors = [
    LinearSegmentedColormap.from_list('vc', [ACCENT2, ACCENT4], N=n_sp)(i / n_sp)
    for i in range(n_sp)
]

# limit of values included -> 99th percentile
MAX_X = np.percentile(np.concatenate(violin_data), 99)

# draw plot
for i, (sp, data) in enumerate(zip(species_order, violin_data)):
    if len(data) < 3:
        continue
        
    y_pos  = sp_to_y[sp]
    color  = viol_colors[i]

    
    x_clip = data[data <= MAX_X]
    
    if len(x_clip) < 3:
        continue
        
    try:
        kde_fn = gaussian_kde(x_clip, bw_method='scott')
        x_eval = np.linspace(0, MAX_X, 300)
        
        density = kde_fn(x_eval)
        density = density / density.max() * 0.45   # # normalize the width

        # make plots symmetrical
        ax3.fill_betweenx(x_eval,
                          y_pos - density, y_pos + density,
                          color=color, alpha=0.55, zorder=2)
        
        # plot both sides
        ax3.plot(y_pos + density, x_eval, color=color, lw=0.7, alpha=0.8, zorder=3)
        ax3.plot(y_pos - density, x_eval, color=color, lw=0.7, alpha=0.8, zorder=3)
        
    except Exception:
        pass

# median and mean markers
for sp in species_order:
    y_pos  = sp_to_y[sp]
    data   = violin_df[violin_df['COMMON NAME'] == sp]['OBSERVATION COUNT'].values
    
    if len(data) == 0:
        continue
        
    median = np.median(data)
    mean   = np.mean(data)

    # line from 0 to median
    ax3.plot([y_pos, y_pos], [0, median],
             color=ACCENT3, lw=1.4, alpha=0.7, zorder=4)
    
    # median marker
    ax3.scatter(y_pos, median,
                s=55, color=ACCENT3, zorder=5,
                edgecolors=BG_DARK, linewidths=0.8)
    
    # mean marker
    ax3.scatter(y_pos, mean,
                s=30, color=ACCENT1, marker='D', zorder=5,
                edgecolors=BG_DARK, linewidths=0.6)

# plot title and styling 
ax3.set_xticks(range(n_sp))
ax3.set_xticklabels(species_order, fontsize=11, color=TEXT_MAIN,
                    rotation=35, ha='right', rotation_mode='anchor')

ax3.set_ylabel('Individual observation count (per checklist)',
               fontsize=12, color=TEXT_DIM)

ax3.set_xlim(-0.6, n_sp - 0.4)
ax3.set_ylim(-2, MAX_X * 1.05)

ax3.yaxis.set_major_formatter(
    ticker.FuncFormatter(
        lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else str(int(max(x, 0)))))

ax3.tick_params(axis='x', colors=TEXT_MAIN, labelsize=11, length=0)
ax3.tick_params(axis='y', colors=TEXT_DIM, labelsize=9)

ax3.grid(axis='y', color=GRID_CLR, linewidth=0.5, alpha=0.65)

ax3.set_axisbelow(True)
for spine in ax3.spines.values():
    spine.set_edgecolor(GRID_CLR)

# legend for plot -> distinguished b/w what is viloin, median, and mean
legend_els = [
    mpatches.Patch(facecolor=ACCENT2, alpha=0.6,
                   label='Distribution of checklist counts (violin)'),
    Line2D([0],[0], marker='o', color='none',
           markerfacecolor=ACCENT3, markersize=8,
           label='Median observation count'),
    Line2D([0],[0], marker='D', color='none',
           markerfacecolor=ACCENT1, markersize=6,
           label='Mean observation count'),
]

ax3.legend(handles=legend_els, loc='upper right',
           fontsize=10.5, framealpha=0.25,
           facecolor=BG_DARK, edgecolor=GRID_CLR, labelcolor=TEXT_MAIN)

# title & subtitle 
ax3.set_title(
    '3. Top Bird Species in Newfoundland   —   Most Frequently Observed Birds of 2025',
    fontsize=20, fontweight='bold', color=ACCENT3, pad=14, loc='left')

ax3.text(
    0.0, -0.15,
    'Violins show the full distribution of per-checklist sighting counts. '
    'Gold dot = median, amber diamond = mean. '
    'Wider violins = a broader spread of flock sizes observed.',
    transform=ax3.transAxes, fontsize=11, color=TEXT_DIM, ha='left')

print('Viz 3 done.')

Viz 3 done.


##### Footer

In [18]:
fig.text(
    0.5, 0.005,
    ' ',
    ha='center', fontsize=10, color=TEXT_DIM, fontstyle='italic')

Text(0.5, 0.005, ' ')

#### Create full infographic & save as PNG file 

In [19]:
OUTPUT = 'infographic.png'
fig.savefig(OUTPUT, dpi=DPI, facecolor=BG_DARK, bbox_inches='tight')
print(f'Saved: {OUTPUT}  ({int(FIG_W*DPI)} × {int(FIG_H*DPI)} px @ {DPI} dpi)')
plt.show()

Saved: infographic.png  (2400 × 3600 px @ 100 dpi)
